# 제조 설비 고장 예측 (Predictive Maintenance)## 프로젝트 개요AI4I 2020 데이터셋을 활용해 제조 설비의 고장을 사전에 예측하는 모델을 구현했습니다.센서 데이터(온도, 회전속도, 토크, 공구마모)를 기반으로 XGBoost 분류 모델을 학습시키고,SHAP 분석으로 고장 원인 변수를 정량적으로 규명했습니다.## 주요 결과- 고장 예측 Recall: 79%- 고장 예측 F1-score: 0.67- 핵심 고장 원인: 공구 마모(Tool wear), 토크(Torque)

In [ ]:
import numpy as npimport pandas as pdimport osfor dirname, _, filenames in os.walk('/kaggle/input'):    for filename in filenames:        print(os.path.join(dirname, filename))

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020/ai4i2020.csv')print(df.shape)print(df.head())

In [ ]:
# 고장 비율 확인print("=== 정상/고장 비율 ===")print(df['Machine failure'].value_counts())print(f"\n고장률: {df['Machine failure'].mean() * 100:.1f}%")# 결측값 확인print(f"\n결측값: {df.isnull().sum().sum()}개")

In [ ]:
from sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom imblearn.over_sampling import SMOTEfrom xgboost import XGBClassifierfrom sklearn.metrics import classification_report# 1. X, y 분리 (실패 세부유형 컬럼(TWF/HDF/PWF/OSF/RNF)은 타겟 정보가 새어들어오므로 제외)X = df.drop(columns=['UDI', 'Product ID', 'Type', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'])y = df['Machine failure']# 2. 학습/테스트 분리 (SMOTE보다 먼저 — test에 합성 샘플이 섞이지 않도록)X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)# 3. 정규화 (train으로만 fit)scaler = StandardScaler()X_train = scaler.fit_transform(X_train)X_test = scaler.transform(X_test)# 4. SMOTE (train에만 적용)sm = SMOTE(random_state=42)X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)# 5. 학습model = XGBClassifier(random_state=42, eval_metric='logloss')model.fit(X_train_sm, y_train_sm)# 6. 결과y_pred = model.predict(X_test)print(classification_report(y_test, y_pred, target_names=['정상', '고장']))

In [ ]:
import shapimport matplotlib.pyplot as plt# SHAP 계산explainer = shap.TreeExplainer(model)shap_values = explainer.shap_values(X_test)# 변수 이름 (그래프에 들어가는 텍스트는 한글 폰트 미탑재 환경(Kaggle 기본)에서# 깨지는 걸 막기 위해 영문으로 표기)feature_names = ['Air temperature [K]', 'Process temperature [K]',                 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']# 시각화shap.summary_plot(shap_values, X_test, feature_names=feature_names, plot_type='bar')plt.tight_layout()plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')plt.show()

In [ ]:
import matplotlib.pyplot as plt# 그래프 텍스트는 전부 영문으로 표기 (한글 라벨은 Kaggle 기본 폰트에서 네모로 깨짐)fig, axes = plt.subplots(2, 2, figsize=(12, 8))fig.suptitle('EDA - Failure Cause Exploration', fontsize=16)# 1. Tool wear distributionaxes[0,0].hist(df[df['Machine failure']==0]['Tool wear [min]'],               alpha=0.7, label='Normal', bins=30, color='steelblue')axes[0,0].hist(df[df['Machine failure']==1]['Tool wear [min]'],               alpha=0.7, label='Failure', bins=30, color='tomato')axes[0,0].set_title('Tool Wear Distribution')axes[0,0].set_xlabel('Tool wear [min]'); axes[0,0].set_ylabel('Count')axes[0,0].legend()# 2. Torque distributionaxes[0,1].hist(df[df['Machine failure']==0]['Torque [Nm]'],               alpha=0.7, label='Normal', bins=30, color='steelblue')axes[0,1].hist(df[df['Machine failure']==1]['Torque [Nm]'],               alpha=0.7, label='Failure', bins=30, color='tomato')axes[0,1].set_title('Torque Distribution')axes[0,1].set_xlabel('Torque [Nm]'); axes[0,1].set_ylabel('Count')axes[0,1].legend()# 3. Rotational speed distributionaxes[1,0].hist(df[df['Machine failure']==0]['Rotational speed [rpm]'],               alpha=0.7, label='Normal', bins=30, color='steelblue')axes[1,0].hist(df[df['Machine failure']==1]['Rotational speed [rpm]'],               alpha=0.7, label='Failure', bins=30, color='tomato')axes[1,0].set_title('Rotational Speed Distribution')axes[1,0].set_xlabel('Rotational speed [rpm]'); axes[1,0].set_ylabel('Count')axes[1,0].legend()# 4. Air temperature distributionaxes[1,1].hist(df[df['Machine failure']==0]['Air temperature [K]'],               alpha=0.7, label='Normal', bins=30, color='steelblue')axes[1,1].hist(df[df['Machine failure']==1]['Air temperature [K]'],               alpha=0.7, label='Failure', bins=30, color='tomato')axes[1,1].set_title('Air Temperature Distribution')axes[1,1].set_xlabel('Air temperature [K]'); axes[1,1].set_ylabel('Count')axes[1,1].legend()plt.tight_layout()plt.show()